# Approximation de rang faible

| Module | Durée | Prérequis |
|---|---|---|
| M8 – Décomposition en valeurs singulières et données | 75 min | M6 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Un service météorologique publie, pour chacune de ses 80 stations, les **normales mensuelles** de température : la température moyenne de chaque mois, sur trente ans. Le fichier `donnees/normales_temperature.csv` contient ces 80 × 12 = 960 valeurs, ainsi que la latitude, l'altitude et la distance à la mer de chaque station. Les données sont simulées, mais elles reproduisent les grands traits du climat d'un pays tempéré.

L'équipe veut créer une petite application qui affiche le climat de chaque station. Un ingénieur remarque que les 80 courbes annuelles se ressemblent beaucoup : un creux en hiver, une bosse en été. Il propose de stocker, pour chaque station, sa **température moyenne annuelle** et **quelques autres nombres** seulement, et de reconstruire les courbes à partir de « profils types » communs.

**Les questions : combien de nombres par station faut-il garder pour que chaque normale reconstruite soit juste à 0,5 °C près ? Que représentent ces profils types ?**

**Activité** : Par quels nombres décririez-vous, vous, le climat d'une ville, si vous n'aviez droit qu'à deux ou trois nombres ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_nombres_par_station = 4      # pour une erreur inférieure à 0,5 °C sur chaque valeur (modifiez)

## ③ Abstraire : une somme de couches de rang 1

On retire d'abord à chaque station sa moyenne annuelle : il reste une matrice $X$ d'**écarts à la moyenne** (une ligne par station, une colonne par mois). La SVD l'écrit comme une somme de **couches** :

$$
X = \sigma_1\,\mathbf{u}_1\mathbf{v}_1^{\mathsf{T}} + \sigma_2\,\mathbf{u}_2\mathbf{v}_2^{\mathsf{T}} + \cdots
$$

Chaque couche $\mathbf{u}_i\mathbf{v}_i^{\mathsf{T}}$ est une matrice de **rang 1** : toutes ses lignes sont proportionnelles au même profil $\mathbf{v}_i^{\mathsf{T}}$ (ici, une courbe sur 12 mois), avec un coefficient par station donné par $\mathbf{u}_i$.

![Schéma : la matrice A est la somme de couches sigma_i u_i v_i transposé, chacune formée d'une colonne fois une ligne ; garder les k premières couches donne la meilleure approximation de rang k](../../assets/schemas/schema-rang-faible.svg)

**Le théorème d'Eckart-Young** affirme que garder les $k$ premières couches donne la **meilleure** approximation de rang $k$ possible, et que l'erreur commise est

$$
\lVert X - X_k \rVert = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \cdots}
$$

(la norme est ici la racine de la somme des carrés de tous les coefficients). Il suffit donc de regarder les valeurs singulières pour savoir combien de couches garder.

**Activité** : Vérifiez que $\begin{pmatrix} 2 & 4 \\ 3 & 6 \end{pmatrix} = \begin{pmatrix} 2 \\ 3 \end{pmatrix}\begin{pmatrix} 1 & 2 \end{pmatrix}$. Quel est le rang de cette matrice ? Combien de nombres suffisent pour la stocker ? Combien en faut-il pour une matrice $80 \times 12$ de rang $k$, en comptant aussi les 80 moyennes annuelles ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les données et leurs valeurs singulières

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

normales = pd.read_csv("donnees/normales_temperature.csv")
mois = ["jan", "fev", "mar", "avr", "mai", "jun", "jul", "aou", "sep", "oct", "nov", "dec"]
T = normales[mois].to_numpy()
moyennes = T.mean(axis=1, keepdims=True)      # moyenne annuelle de chaque station : stockée à part
X = T - moyennes                              # écarts à la moyenne annuelle
print("Taille :", X.shape)

U, s, Vt = np.linalg.svd(X, full_matrices=False)
# Le signe des vecteurs singuliers est arbitraire : on l'oriente pour faciliter la lecture
for i, mois_repere in [(0, 6), (1, 9)]:            # profil 1 positif en juillet, profil 2 positif en octobre
    if Vt[i, mois_repere] < 0:
        U[:, i], Vt[i] = -U[:, i], -Vt[i]
part = s ** 2 / np.sum(s ** 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(X.T, color=LIOUAI["navy"], alpha=0.25)
axes[0].set_xticks(range(12), mois); axes[0].set_ylabel("écart à la moyenne annuelle (°C)")
axes[0].set_title("Les 80 courbes, moyenne annuelle retirée")
axes[1].semilogy(range(1, 13), s, "o-", color=LIOUAI["navy"])
axes[1].set_xlabel("rang i"); axes[1].set_ylabel("σi (échelle log)"); axes[1].set_title("Valeurs singulières")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Part de chaque couche dans la somme des carrés :", ", ".join(f"{p:.3%}" for p in part[:5]))

### L'erreur selon le nombre de couches gardées

On mesure l'erreur en °C, comme une racine de l'erreur quadratique moyenne (module M6), et on la compare à la formule d'Eckart-Young.

In [ ]:
def approximation(U, s, Vt, k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k]

def stockage(k):
    return 80 + k * (80 + 12 + 1)             # moyennes annuelles + k couches

print(f"{'k':>2s} {'erreur moyenne (°C)':>20s} {'par la formule':>15s} {'erreur max (°C)':>16s} {'nombres stockés':>16s}")
for k in range(0, 6):
    Xk = approximation(U, s, Vt, k)
    reqm = np.sqrt(np.mean((X - Xk) ** 2))
    formule = np.sqrt(np.sum(s[k:] ** 2) / X.size)
    print(f"{k:2d} {reqm:20.3f} {formule:15.3f} {np.abs(X - Xk).max():16.2f} {stockage(k):16d}")
print(f"Données complètes : {T.size} nombres")

**Activité** : Que représente la ligne $k = 0$ ? À partir de quel $k$ l'erreur moyenne passe-t-elle sous 0,5 °C ? Et l'erreur maximale, qui est le critère de l'équipe ? Au-delà, l'erreur baisse beaucoup plus lentement : que reste-t-il dans les données, selon vous ? Est-ce utile de le garder ?

Indiquez vos réponses et justifiez-les.

### Que sont les profils types ?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for i, ax in enumerate(axes):
    ax.plot(range(12), Vt[i], "o-", color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"]][i])
    ax.axhline(0, color=LIOUAI["gris"]); ax.set_xticks(range(12), mois, fontsize=8)
    ax.set_title(f"profil v{i + 1} (σ{i + 1} = {s[i]:.1f})"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Le signe d'un vecteur singulier est arbitraire : `svd` peut renvoyer $-\mathbf{u}_i$ et $-\mathbf{v}_i$ au lieu de $\mathbf{u}_i$ et $\mathbf{v}_i$. Seul leur produit compte ; c'est pourquoi on a choisi les signes plus haut.

**Activité** : Décrivez les deux premiers profils en mots. Lequel représente le contraste entre l'hiver et l'été ? Que peut bien représenter le deuxième, négatif au printemps et positif à l'automne ? Le troisième ressemble-t-il à quelque chose ?

Indiquez vos réponses et justifiez-les.

### Relier les coefficients aux stations

Chaque station a sa moyenne annuelle et un coefficient sur chaque profil : $\sigma_i u_{si}$. Comparons-les à la géographie.

In [ ]:
grandeurs = {"moyenne annuelle": moyennes[:, 0], "coefficient 1": U[:, 0] * s[0], "coefficient 2": U[:, 1] * s[1]}
geographie = {"latitude": "latitude (°N)", "altitude_m": "altitude (m)", "distance_mer_km": "distance à la mer (km)"}
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for i, (nom_g, g) in enumerate(grandeurs.items()):
    for j, (colonne, nom) in enumerate(geographie.items()):
        ax = axes[i, j]
        ax.plot(normales[colonne], g, "o", ms=3, color=LIOUAI["navy"])
        r = np.corrcoef(normales[colonne], g)[0, 1]
        ax.set_title(f"{nom_g} / {colonne} : r = {r:+.2f}", fontsize=9)
        if i == 2:
            ax.set_xlabel(nom)
        ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Reconstruire une station

In [ ]:
station = int(np.argmax(normales["distance_mer_km"]))      # la station la plus continentale
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(T[station], "ko", label="normales réelles")
for k, couleur in zip((0, 1, 2), [LIOUAI["gris"], LIOUAI["jaune"], LIOUAI["vert"]]):
    ax.plot(moyennes[station, 0] + approximation(U, s, Vt, k)[station], color=couleur, lw=2,
            label=f"moyenne + {k} couche(s)")
ax.set_xticks(range(12), mois); ax.set_ylabel("°C"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"Station {normales['station'][station]} ({normales['distance_mer_km'][station]} km de la mer)")
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
k_retenu = next(k for k in range(0, 13) if np.abs(X - approximation(U, s, Vt, k)).max() < 0.5)
print(f"Votre pari : {pari_nombres_par_station} nombres par station ; "
      f"il en faut {1 + k_retenu} (la moyenne annuelle et {k_retenu} coefficients)")
print(f"Stockage : {stockage(k_retenu)} nombres au lieu de {T.size}, soit {stockage(k_retenu) / T.size:.0%}")

**Activité** :

1. Rédigez la réponse pour l'équipe : combien de nombres par station, quelle erreur, quel gain de stockage.
2. Donnez un nom parlant aux deux profils retenus, et dites de quelle caractéristique géographique dépendent la moyenne annuelle et chaque coefficient.
3. Le gain de stockage est modeste ici (80 stations). Comment évolue-t-il avec 5 000 stations ? avec des normales quotidiennes (365 colonnes) ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Que perd-on ?** Les couches au-delà de la deuxième ont toutes à peu près la même valeur singulière. Que suggère ce « plateau » sur leur nature ? Est-ce une perte ?
- **Les données sont des normales**, donc déjà des moyennes. La même méthode marcherait-elle aussi bien sur les températures d'une année particulière ?
- **Généraliser.** Une nouvelle station ouvre. On connaît ses 12 normales. Sans recalculer la SVD, on obtient ses coefficients en **projetant** ses écarts à la moyenne $\mathbf{x}$ sur les profils (module M6) : $\mathbf{c} = V_2^{\mathsf{T}}\mathbf{x}$, car les profils sont orthonormés. Écrivez une fonction `coder(x, Vt, k)` et une fonction `decoder(c, Vt)`, puis testez-les sur une station retirée des données avant le calcul de la SVD.

In [ ]:
def coder(x, Vt, k):
    # À compléter
    ...

def decoder(c, Vt):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def coder(x, Vt, k):
    return Vt[:k] @ x                 # coordonnées sur les k premiers profils (orthonormés)

def decoder(c, Vt):
    return Vt[:len(c)].T @ c

nouvelle = 17
X_sans = np.delete(X, nouvelle, axis=0)            # on calcule les profils sans cette station
_, _, Vt_sans = np.linalg.svd(X_sans, full_matrices=False)
for i, mois_repere in [(0, 6), (1, 9)]:            # même orientation des signes que plus haut
    if Vt_sans[i, mois_repere] < 0:
        Vt_sans[i] = -Vt_sans[i]
c = coder(X[nouvelle], Vt_sans, 2)
erreur = np.abs(X[nouvelle] - decoder(c, Vt_sans)).max()
print(f"Station {normales['station'][nouvelle]} : moyenne {moyennes[nouvelle, 0]:.2f} °C et coefficients {c.round(2)} ; "
      f"erreur maximale de reconstruction {erreur:.2f} °C")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Relier les traits du résultat à leur sens réel": 1,
    "Décider si les résultats suffisent pour avancer ou abandonner": 1,
    "Estimer la solution avant de calculer": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
    "Lister les sources d'erreur liées aux limites du calcul": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "8.2-rang-faible", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")